In [1]:
%pip install torch torchvision scikit-learn matplotlib pillow

  Using cached torch-2.10.0-cp314-cp314-win_amd64.whl.metadata (31 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached scipy-1.17.1-cp314-cp314-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
   ---------------------------------------- 0.0/113.8 MB ? eta -:--:--
   ---------------------------------------- 1.0/113.8 MB 6.1 MB/s eta 0:00:19
    --------------------------------------- 2.1/113.8 MB 5.9 MB/s eta 0:00:19
   - -------------------------------------- 3.4/113.8 MB 5.8 MB/s eta 0:00:19
   - -------------------------------------- 4.5/113.8 MB 5.8 MB/s eta 0:00:19
   - -------------------------------------- 5.5/113.8 MB 5.6 MB/s eta 0:00:20
   -- ------------------------------------- 5.8/113.8 MB 5.0 MB/s eta 0:00:22
   -- ------------------------------------- 6.6/113.8 MB 4.7 MB/s eta 0:00:23
   -- ------------------------------------- 7.6/113.8 MB 4.7 MB/s eta 0:00:23
   --- ----------------------------

In [ ]:
# MULTI-CANCER CLASSIFICATION

import os
import shutil
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from PIL import Image

random.seed(42)
torch.manual_seed(42)

# PATHS
SOURCE_DIR = "C:/Files/IISc/AI in Health Care/Multi Cancer/Multi Cancer/Input"
DEST_DIR = "C:/Files/IISc/AI in Health Care/Multi Cancer/Multi Cancer/data_split_kidney2"


# number of images per class 
MAX_IMAGES_PER_CLASS = 500

SPLITS = {"train": 0.7, "val": 0.15, "test": 0.15}


In [ ]:
# DATA SPLITTING (SAMPLED)
def split_dataset():

    if os.path.exists(DEST_DIR):
        print("Split already exists — skipping...")
        return

    print("Creating sampled dataset...")

    for cancer_type in os.listdir(SOURCE_DIR):

        cancer_path = os.path.join(SOURCE_DIR, cancer_type)
        if not os.path.isdir(cancer_path):
            continue

        for subtype in os.listdir(cancer_path):

            class_path = os.path.join(cancer_path, subtype)
            if not os.path.isdir(class_path):
                continue

            images = os.listdir(class_path)

            # SAMPLE SMALL DATASET
            random.shuffle(images)
            images = images[:MAX_IMAGES_PER_CLASS]

            n = len(images)
            train_end = int(n * SPLITS["train"])
            val_end = train_end + int(n * SPLITS["val"])

            split_map = {
                "train": images[:train_end],
                "val": images[train_end:val_end],
                "test": images[val_end:]
            }

            for split, files in split_map.items():

                dest_folder = os.path.join(DEST_DIR, split, subtype)
                os.makedirs(dest_folder, exist_ok=True)

                for f in files:
                    shutil.copy2(
                        os.path.join(class_path, f),
                        os.path.join(dest_folder, f)
                    )

    print("Dataset split completed!")

split_dataset()



Creating sampled dataset...
Dataset split completed!


In [ ]:
# DATA LOADING
IMG_SIZE = 224
BATCH_SIZE = 8

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(f"{DEST_DIR}/train", transform=transform)
val_dataset   = datasets.ImageFolder(f"{DEST_DIR}/val", transform=transform)
test_dataset  = datasets.ImageFolder(f"{DEST_DIR}/test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2, pin_memory=True)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print("Classes:", train_dataset.classes)


Classes: ['kidney_normal', 'kidney_tumor']


In [ ]:
# MODEL — Vision Transformer
from torchvision.models import vit_b_16, ViT_B_16_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)

# Freeze backbone
for param in model.parameters():
    param.requires_grad = False

num_classes = len(train_dataset.classes)

# Replace classifier head
model.heads.head = nn.Linear(
    model.heads.head.in_features,
    num_classes
)

model = model.to(device)


In [ ]:
# TRAIN SETUP
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.heads.head.parameters(),
    lr=0.001
)

In [ ]:
# TRAINING LOOP
def train_model(epochs=10):

    for epoch in range(epochs):

        model.train()
        running_loss = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss:.4f}")

train_model()

c:\Files\IISc\AI in Health Care\Multi Cancer\Multi Cancer\.venv\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 1/10 | Loss: 34.7948
Epoch 2/10 | Loss: 18.3884
Epoch 3/10 | Loss: 12.7142
Epoch 4/10 | Loss: 9.8044
Epoch 5/10 | Loss: 8.0946
Epoch 6/10 | Loss: 6.5735
Epoch 7/10 | Loss: 5.6012
Epoch 8/10 | Loss: 4.8447
Epoch 9/10 | Loss: 4.3801
Epoch 10/10 | Loss: 3.6794


In [ ]:
# EVALUATION
def evaluate(loader):

    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

print("Validation Accuracy:", evaluate(val_loader))
print("Test Accuracy:", evaluate(test_loader))



Validation Accuracy: 0.98
Test Accuracy: 0.9866666666666667


In [ ]:
# CLASS METADATA (Subclass + Description)

CLASS_INFO = {
    # "all_benign": ("Benign", "Non-cancerous, healthy cells"),
    # "all_early": ("Early", "Early stages of leukemia"),
    # "all_pre": ("Pre", "Pre-stage abnormal cells"),
    # "all_pro": ("Pro", "Advanced leukemia cells"),

    # "brain_glioma": ("Glioma", "Most common brain tumor"),
    # "brain_menin": ("Meningioma", "Tumors affecting brain membranes"),
    # "brain_tumor": ("Pituitary Tumor", "Tumors affecting the pituitary gland"),

    # "breast_benign": ("Benign", "Non-cancerous breast tissues"),
    # "breast_malignant": ("Malignant", "Cancerous breast tissues"),

    # "cervix_dyk": ("Dyskeratotic", "Abnormal cell growth"),
    # "cervix_koc": ("Koilocytotic", "Cells showing viral infection changes (HPV)"),
    # "cervix_mep": ("Metaplastic", "Cells changed from one type to another (precancerous)"),
    # "cervix_pab": ("Parabasal", "Immature squamous cells"),
    # "cervix_sfi": ("Superficial-Intermediate", "More mature squamous cells"),

    "kidney_normal": ("Normal", "Healthy kidney tissues"),
    "kidney_tumor": ("Tumor", "Tumor-affected kidney tissues"),

    # "colon_aca": ("Colon Adenocarcinoma", "Cancerous cells of the colon"),
    # "colon_bnt": ("Colon Benign Tissue", "Healthy colon tissues"),

    # "lung_aca": ("Lung Adenocarcinoma", "Cancerous lung cells"),
    # "lung_bnt": ("Lung Benign Tissue", "Healthy lung tissues"),
    # "lung_scc": ("Lung Squamous Cell Carcinoma", "Aggressive lung cancer type"),

    # "lymph_cll": ("Chronic Lymphocytic Leukemia", "Slow-progressing blood cancer"),
    # "lymph_fl": ("Follicular Lymphoma", "Slow-growing non-Hodgkin lymphoma"),
    # "lymph_mcl": ("Mantle Cell Lymphoma", "Aggressive lymphoma"),

    # "oral_normal": ("Normal", "Healthy oral tissues"),
    # "oral_scc": ("Oral Squamous Cell Carcinoma", "Cancerous oral cells"),
}

In [ ]:
# TOP CANCER PREDICTION WITH PROBABILITIES
import torch.nn.functional as F


def predict_top3(image_path):

    img = Image.open(image_path).convert("RGB")
    img = transform(img).unsqueeze(0).to(device)

    model.eval()

    with torch.no_grad():
        outputs = model(img)

        # Convert logits → probabilities
        probs = F.softmax(outputs, dim=1)[0]

        # Top 3 predictions
        top_probs, top_indices = torch.topk(probs, k=2)

    results = []

    print("\n==============================")
    print("   AI CANCER DIAGNOSIS REPORT")
    print("==============================\n")

    for rank, (prob, idx) in enumerate(zip(top_probs, top_indices), start=1):

        class_name = train_dataset.classes[idx.item()]

        subclass, description = CLASS_INFO.get(
            class_name,
            ("Unknown", "Description unavailable")
        )

        probability = prob.item() * 100

        result = {
            "rank": rank,
            "class": class_name,
            "subclass": subclass,
            "description": description,
            "probability": probability,
        }

        results.append(result)

        print(f"{rank}. Prediction: {class_name}")
        print(f"   Subclass   : {subclass}")
        print(f"   Probability: {probability:.2f}%")
        print(f"   Description: {description}")
        print()

    return results

In [9]:
predict_top3(
"C:/Files/IISc/AI in Health Care/Multi Cancer/kidney_tumor_0113.jpg"
)


   AI CANCER DIAGNOSIS REPORT

1. Prediction: kidney_tumor
   Subclass   : Tumor
   Probability: 99.83%
   Description: Tumor-affected kidney tissues

2. Prediction: kidney_normal
   Subclass   : Normal
   Probability: 0.17%
   Description: Healthy kidney tissues



[{'rank': 1,
  'class': 'kidney_tumor',
  'subclass': 'Tumor',
  'description': 'Tumor-affected kidney tissues',
  'probability': 99.8325765132904},
 {'rank': 2,
  'class': 'kidney_normal',
  'subclass': 'Normal',
  'description': 'Healthy kidney tissues',
  'probability': 0.16742207808420062}]